## Data Loader

In [1]:
import tiktoken

In [ ]:
# Create ticktoken BPE tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
# Read text file
with open("../data/deu_wikipedia_2021_1M-sentences.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
# Tokenize text
txt = tokenizer.encode(raw_text)
print("Number of tokens: ", len(txt))

Number of tokens:  41663341


In [ ]:
# Take a sample
sample = txt[1010:1055]
print(f"Token IDs: ", sample)
print(f"Tokens: ")

for token_id in sample:
    print(tokenizer.decode([token_id]), end="|")

Token IDs:  [6733, 24417, 260, 353, 4587, 1168, 2464, 1350, 85, 9101, 75, 6122, 2150, 356, 13254, 2853, 564, 252, 6935, 23048, 268, 447, 250, 288, 283, 559, 69, 289, 259, 11, 288, 562, 4656, 520, 324, 83, 21303, 9101, 4743, 488, 1976, 84, 30362, 312, 9324]
Tokens: 
 Die| Vert|re|ter| der| Z|ivil|be|v|ö|l|ker|ung| we|isen| den| �|�|Comm|endant|en|�|�| d|ar|au|f| h|in|,| d|ass| die| St|ad|t| unm|ö|gl|ich| z|u| verte|id|igen|

In [ ]:
tokenizer = 4 # How many tokens do we look at when predicting the next token?
x = txt[:tokenizer] # Input tokens
y = txt[1:tokenizer+1] # Target token (the one we want to predict)
print("x: ", x)
print("y:     ", y)

x:  [26, 657, 11, 22]
y:      [657, 11, 22, 1041]


In [ ]:
print("[input token IDs] -> target token ID")
print("-------------------------------")

for i in range(1, tokenizer+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{context} -> {desired}")

[input token IDs] -> target token ID
-------------------------------
[6733] -> 24417
[6733, 24417] -> 260
[6733, 24417, 260] -> 353
[6733, 24417, 260, 353] -> 4587


In [ ]:
print("[input tokens] -> target token")
print("-------------------------------")

for i in range(1, tokenizer+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{tokenizer.decode(context)} -> {tokenizer.decode([desired])}")

[input tokens] -> target token
-------------------------------
 Die ->  Vert
 Die Vert -> re
 Die Vertre -> ter
 Die Vertreter ->  der


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]

            self.input_ids.append(input_chunk)
            self.target_ids.append(target_chunk)

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.input_ids)

    def __getitem__(self, idx):
        """Returns a single sample from the dataset."""
        return self.input_ids[idx], self.target_ids[idx]

In [71]:
# Demonstration of GPTDataset1 with dummy data.
example_token_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

max_length = 4
stride = 1

print("[input token IDs chunk] -> [target token IDs chunk]")
print("---------------------------------------")

for i in range(0, len(example_token_ids) - max_length, stride):
    input_chunk = example_token_ids[i:i+max_length]
    target_chunk = example_token_ids[i+1:i+max_length+1]
    print(f"{input_chunk} -> {target_chunk}")

[input token IDs chunk] -> [target token IDs chunk]
---------------------------------------
[0, 1, 2, 3] -> [1, 2, 3, 4]
[1, 2, 3, 4] -> [2, 3, 4, 5]
[2, 3, 4, 5] -> [3, 4, 5, 6]
[3, 4, 5, 6] -> [4, 5, 6, 7]
[4, 5, 6, 7] -> [5, 6, 7, 8]
[5, 6, 7, 8] -> [6, 7, 8, 9]
[6, 7, 8, 9] -> [7, 8, 9, 10]
[7, 8, 9, 10] -> [8, 9, 10, 11]
[8, 9, 10, 11] -> [9, 10, 11, 12]
[9, 10, 11, 12] -> [10, 11, 12, 13]
[10, 11, 12, 13] -> [11, 12, 13, 14]
[11, 12, 13, 14] -> [12, 13, 14, 15]
[12, 13, 14, 15] -> [13, 14, 15, 16]
[13, 14, 15, 16] -> [14, 15, 16, 17]
[14, 15, 16, 17] -> [15, 16, 17, 18]
[15, 16, 17, 18] -> [16, 17, 18, 19]
